In [7]:
# load packages
from pyfiles.preamble import *
setup_notebook() 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create Scenario

This notebook builds EnergyPLAN input files from a reference scenario by overriding specific parameters. It is the single source of truth for all scenario parameters — the other notebooks (`2_get_output`, `3_optimiser`) read directly from the files produced here.

**Workflow**
1. Pick a reference `.txt` file to build on
2. Define shared parameter changes (`base_params`)
3. Optionally define case-specific overrides (`base_case_params`, `shock_case_params`) and loop over cases

Browse `pyfiles/input_variables.py` for available parameter names. Decimal values are written as fractions (e.g. `311/10`) to keep the intent readable.

### 1. Reference file

Pick a reference `.txt` file to base the scenario on. All parameters not explicitly overridden below will keep their values from this file. Use `initalize.txt` for a blank starting point.

In [8]:
ref_path = ep_run.INPUT_DIR / "IDA2045_Final.txt"

### 2. Single scenario

Define all parameter overrides in `base_params` and save to a single output file.

*Shared parameters — applied to all cases:*

In [9]:
base_params = {

    # variables --------------- # value -------------------------
    # demand
    'Input_el_demand_Twh'       : 311/10,   # decimals as fractions, run faster
    'Input_add_el_TWh'          : 156/10,    

    # production
    'input_RES1_capacity'       : 5150,    
    'input_RES3_capacity'       : 25134,     

    # nuclear
    'input_nuclear_eff'         : 1,       
    'input_Nuclear_factor'      : 9/10,    
    'input_fuel_price[12]'      : 259/100,
    'input_Nuclear_partload'    : 3/10,                              

    # costs
    'input_Inv_Nuclear'         : 8,         
    'input_Period_Nuclear'      : 60,         
    'input_FOM_Nuclear'         : 193/100,     

    'input_Inv_WindOffshore'    : 25/10,   
    'input_Period_WindOffshore' : 30,       
    'input_FOM_WindOffshore'    : 191/100,    
}

*Save to file:*

In [10]:
save_scenario(
    ref_path = ref_path, 
    out_path = ep_run.INPUT_DIR / "example.txt", 
    params = base_params,
)

Written: c:\Users\LinusLindquist\OneDrive - e-tank\Skrivebord\ZipEnergy\energyPlan Data\Data\example.txt


### 3. Base and shock scenario pair

Extend `base_params` with case-specific overrides, then loop to produce one file per case. `base_case_params` and `shock_case_params` only need to contain the parameters that differ between cases.

*Case-specific overrides — these are merged on top of `base_params`:*

In [11]:
    # variables --------------- # value -------------------------

base_case_params = {
    'input_nuclear_cap'         : 0,         
    'input_RES2_capacity'       : 8287,  
    'input_keol_reg'            : 234570000, 

    'input_cap_pp2_el'          : 2363,     
    'input_cap_ELTtrans_el'     : 4621,
    'input_H2storage_trans_cap' : 32,         
    'input_storage_pump_cap'    : 120,   
}

shock_case_params = {
    'input_nuclear_cap'         : 1000,  
    'input_RES2_capacity'       : 8287-1760,   
    'input_keol_reg'            : 923457000,    
    'input_cshp_th_gr3'         : 128/100,    

    'input_cap_ELTtrans_el'     : 4304,     
    'input_H2storage_trans_cap' : 32,   
    'input_storage_pump_cap'    : 112,   
}

In [12]:
for case in ('base', 'shock'):
    out_path = ep_run.INPUT_DIR / f"example_{case}.txt"
    params = build_params(case, base_params, base_case_params, shock_case_params)

    save_scenario(
        ref_path = ref_path, 
        out_path = out_path, 
        params = params,
    )

Written: c:\Users\LinusLindquist\OneDrive - e-tank\Skrivebord\ZipEnergy\energyPlan Data\Data\example_base.txt
Written: c:\Users\LinusLindquist\OneDrive - e-tank\Skrivebord\ZipEnergy\energyPlan Data\Data\example_shock.txt
